In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np

# Qwen only, four different shuffles, with length, wiki and conspi ----> added slopes for different shuffles
###################################################

wiki = pd.read_csv("results_wikipedia.csv")
cons = pd.read_csv("results_conspiracy.csv")

wiki = wiki[wiki["model_name"].str.contains("Qwen")].copy()
cons = cons[cons["model_name"].str.contains("Qwen")].copy()

wiki["content"] = "wikipedia"
cons["content"] = "conspiracy"

data = pd.concat([wiki, cons], ignore_index=True)

data = data[data["context_type"].isin(["clean", "meani", "wordd", "chara"])]

data["content"] = data["content"].astype("category")
data["content"] = data["content"].cat.set_categories(["wikipedia", "conspiracy"], ordered=True)

data["context_type"] = data["context_type"].astype("category")
data["context_type"] = data["context_type"].cat.set_categories(["clean", "meani", "wordd", "chara"],ordered=True)

model = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="context_title",
    re_formula="~context_type"
)

result = model.fit(reml=True)
print(result.summary())



/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                            Mixed Linear Model Regression Results
Model:                         MixedLM             Dependent Variable:             accuracy  
No. Observations:              8016                Method:                         REML      
No. Groups:                    2004                Scale:                          0.0002    
Min. group size:               4                   Log-Likelihood:                 19904.1419
Max. group size:               4                   Converged:                      Yes       
Mean group size:               4.0                                                           
---------------------------------------------------------------------------------------------
                                                  Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------------
Intercept                                          0.563    0.001 655.522 0.000  0.561  

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# true vs fake, clean and meani, all models
####################################################################

true_results = pd.read_csv("results_isot_true.csv")
fake_results = pd.read_csv("results_isot_fake.csv")

true_results = true_results[true_results['context_type'].isin(['clean', 'meani'])].copy()
fake_results = fake_results[fake_results['context_type'].isin(['clean', 'meani'])].copy()

true_results['content'] = 'isot_true'
fake_results['content'] = 'isot_fake'

data = pd.concat([true_results, fake_results], ignore_index=True)
data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['isot_true', 'isot_fake'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

############################################

model_intercept = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="model_name",
    vc_formula={"context": "0 + C(context_title)"}
)

result_intercept = model_intercept.fit(reml=True, method="lbfgs")

print("random intercepts only:")
print(result_intercept.summary())

#######################################

/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


random intercepts only:
                        Mixed Linear Model Regression Results
Model:                      MixedLM           Dependent Variable:           accuracy  
No. Observations:           16000             Method:                       REML      
No. Groups:                 4                 Scale:                        0.0008    
Min. group size:            4000              Log-Likelihood:               25162.6434
Max. group size:            4000              Converged:                    Yes       
Mean group size:            4000.0                                                    
--------------------------------------------------------------------------------------
                                           Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------------------
Intercept                                   0.539    0.001 526.149 0.000  0.537  0.541
content[T.isot_fake]                       -

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, models as fixed effect
###############################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)

data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

############################################

model_by_llm = smf.mixedlm(
    "accuracy ~ content * context_type * model_name + scale(text_length)",
    data,
    groups="model_name",
    vc_formula={"context": "0 + C(context_title)"}
)

result_by_llm = model_by_llm.fit(reml=True, method="lbfgs")
print(result_by_llm.summary())




/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                                                 Mixed Linear Model Regression Results
Model:                                        MixedLM                           Dependent Variable:                           accuracy  
No. Observations:                             16032                             Method:                                       REML      
No. Groups:                                   4                                 Scale:                                        0.0006    
Min. group size:                              4008                              Log-Likelihood:                               31021.2638
Max. group size:                              4008                              Converged:                                    Yes       
Mean group size:                              4008.0                                                                                    
-------------------------------------------------------------------------------------------